In [ ]:
    # Notebook: Preprocessing e Preparazione Dati per il Machine Learning
    #**Obiettivo:** Utilizzare il dataset di feature estratto da pgAdmin 
    #(`customer_features.csv`) per creare una pipeline di preprocessing e 
    #preparare i dati per l'addestramento di un modello di Machine Learning. 
    #**Questo notebook si ferma prima del training del modello.**"
  
    
    ### 1. Import delle Librerie"
 
    import pandas as pd
    import numpy as n
    import matplotlib.pyplot as plt
    import seaborn as sns
    from datetime import datetime
    
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    
    import warnings
    warnings.filterwarnings("ignore")

In [ ]:
                        
    ### 2. Caricamento del Dataset
    #"Carichiamo il file `customer_features.csv` che hai esportato da pgAdmin."
try:
     df = pd.read_csv("python.1/feature_engineer/customer_features.csv")
     print("Dataset caricato con successo!")
     display(df.head())
except FileNotFoundError:
    print("ERRORE: File 'customer_features.csv' non trovato. Assicurati di averlo esportato da pgAdmin e salvato nella stessa directory di questo notebook.")

In [ ]:
### 3. Feature Engineering Aggiuntivo e Definizione del Target\n",
#"Creiamo le ultime feature derivate e la nostra variabile target `is_churn`."
# Conversione delle colonne di data"
df["data_ultimo_ordine"] = pd.to_datetime(df["data_ultimo_ordine"])
df["data_primo_ordine"] = pd.to_datetime(df["data_primo_ordine"])

# Data di riferimento per il calcolo (es. il giorno dopo l'ultimo ordine registrato nel dataset)
snapshot_date = df["data_ultimo_ordine"].max() + pd.DateOffset(days=1)

# Creazione feature di Recency e Tenure
df["recency"] = (snapshot_date - df["data_ultimo_ordine"]).dt.days
df["tenure"] = (snapshot_date - df["data_primo_ordine"]).dt.days
 
# Definizione della variabile Target
# Un cliente è considerato 'churned' se non ha fatto acquisti da più di 90 giorni
df["is_churn"] = (df["recency"] > 90).astype(int)

print("Distribuzione della variabile target:")
print(df["is_churn"].value_counts(normalize=True))

display(df.head())

In [ ]:
# 4. Selezione delle Feature e Split Train/Test
# Selezioniamo le colonne che useremo come feature (X) e la nostra variabile target (y). 
# Poi, dividiamo il dataset.
# Selezioniamo le feature numeriche e categoriche
numeric_features = ["recency", "tenure", "numero_ordini", "fatturato_totale", "scontrino_medio", "totale_prodotti_acquistati", "num_prodotti_distinti", "num_categorie_distinte"]
categorical_features = ["regione"]

# Definiamo X e y"
X = df[numeric_features + categorical_features]
y = df["is_churn"]

# Suddivisione in set di training e test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dimensioni X_train: {X_train.shape}")
print(f"Dimensioni X_test: {X_test.shape}")

In [ ]:
### 5. Creazione della Pipeline di Preprocessing
#Costruiamo una pipeline con `ColumnTransformer` per applicare trasformazioni diverse alle colonne numeriche e categoriche.

In [ ]:
# Pipeline numeriche: imputazione + scaling
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Pipeline categoriche: imputazione + one-hot encoding
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# ColumnTransformer: combina le due pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [ ]:
### 6. Applicazione del Preprocessing
# Applichiamo la pipeline di preprocessing ai dati di training e test.


In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Dimensioni X_train dopo il preprocessing: {X_train_processed.shape}")
print(f"Dimensioni X_test dopo il preprocessing: {X_test_processed.shape}")


In [ ]:
### 7. Conclusione
# Abbiamo costruito una pipeline che preprocessa i dati e li prepara per un modello di ML.

# I dati `X_train_processed`, `y_train`, `X_test_processed`, `y_test` sono ora pronti per essere usati nel training e nella valutazione del modello.


In [ ]:
### 7.0 Dati pronti per il Machine Learning

# Osserviamo un’estrazione delle feature dopo il preprocessing.
# Questi sono i dati che verranno realmente forniti al modello.


In [ ]:
X_train_ml = pd.DataFrame(
    X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed
)

display(X_train_ml.head())


In [ ]:
### 7.1 Effetto dello scaling sulle feature numeriche

#Confrontiamo la distribuzione delle feature numeriche prima e dopo lo scaling.


In [ ]:
X[numeric_features].boxplot(rot=90, figsize=(12,5))
plt.title("Feature numeriche PRIMA dello scaling")
plt.show()

X_scaled = pd.DataFrame(
    preprocessor.named_transformers_["num"]
    .fit_transform(X_train[numeric_features]),
    columns=numeric_features
)

X_scaled.boxplot(rot=90, figsize=(12,5))
plt.title("Feature numeriche DOPO lo scaling")
plt.show()


In [ ]:
### 7.2 Trasformazione delle variabili categoriche (One-Hot Encoding)

# Osserviamo come la variabile `regione` viene trasformata in più colonne numeriche.


In [ ]:
ohe = preprocessor.named_transformers_["cat"]["onehot"]
cat_feature_names = ohe.get_feature_names_out(categorical_features)

cat_feature_names


In [ ]:
### 7.3 Relazione tra una feature chiave e il churn

# Analizziamo la relazione tra `recency` e la variabile target `is_churn`.


In [ ]:
sns.boxplot(x="is_churn", y="recency", data=df)
plt.title("Recency vs Churn")
plt.show()
